### `rk4.ipynb`
*Edited: Sept 21, 2026* <br/>
This notebooks implements the 4th order explicit Runge-Kutta ODE solver.

In [1]:
"""
Fourth-order explicit Runge-Kutta method for solving du/dt = f(u,p,t),  u(t0) = u0.

Note: This implementation uses a fixed step size `dt`. 

PARAMETERS
----------
f :: the right-hand side of the ODE du/dt = f(u,p,t)
u0 :: the initial value of the solution
tspan :: tuple of the initial and final times for integration
p :: parameter for the ODE (if required); either a scalar or a tuple
dt :: the step size for the integration 

RETURNS
-------
sol :: NamedTuple
    sol.u :: solution vector; u[i] = solution at i-th time step
    sol.t :: vector of times at which the solution is computed
"""
function rk4(f::F, u0::U, tspan::NTuple{2,Float64}, p = nothing; dt::Float64) where {F, U}
   
    #Input validation
    all(isfinite, tspan) || throw(ArgumentError("Time interval must be finite."))
    tspan[1] < tspan[2] || throw(ArgumentError("Initial time must be strictly less than final time."))
    isfinite(dt) || throw(ArgumentError("`dt` must be finite."))
    dt > 0 || throw(ArgumentError("`dt` must be strictly positive."))

    #Compute number of steps and time points
    t0, tf = tspan                                             #Endpoints of time interval
    num_steps = Int(floor((tf - t0)/dt))                       #Number of steps to be taken by the ODE solver 
    t = collect(range(t0, step = dt, length = num_steps + 1))  #Time values at which to record the solution 
                                                            
    #Initialize array to store solution
    u0 = float.(u0)   
    u = [zero(u0) for _ in 1:num_steps + 1]            
    u[1] = copy(u0)       

    h = dt #Prefer to use 'h' like in most textbooks

    #The RK4 algorithm    
    for n = 1:num_steps     #n+1 = 2,3,...,nsteps+1  (remember: length(u) = N + 1) 
        
        k1 = f(u[n]           ,  p,  t[n])        
        k2 = f(u[n] + h * k1/2,  p,  t[n] + h/2)
        k3 = f(u[n] + h * k2/2,  p,  t[n] + h/2)
        k4 = f(u[n] + h * k3  ,  p,  t[n] + h)
        
        u[n+1] = u[n] + (h/6)*(k1 + 2*k2 + 2*k3 + k4)
    end

    sol = (u = u, t = t, dt = dt)
    return sol 
end

rk4